In [1]:


def parse_numeric_log(log_records):
    valid = []
    corrupted_count = 0

    for record in log_records:
        try:
            parts = record.split(",")

            if len(parts) != 2:
                raise ValueError

            txn_id = parts[0].strip()
            value = float(parts[1].strip())

            valid.append({
                "id": txn_id,
                "value": value
            })

        except (ValueError, IndexError):
            corrupted_count += 1

    return {
        "valid": valid,
        "corrupted_count": corrupted_count
    }


log_records = [
    "TXN101, 145.50",
    "TXN102, invalid_num",
    "TXN103, 300.00",
    "CORRUPTED_LINE",
    "TXN104, 82.25"
]

print(parse_numeric_log(log_records))

{'valid': [{'id': 'TXN101', 'value': 145.5}, {'id': 'TXN103', 'value': 300.0}, {'id': 'TXN104', 'value': 82.25}], 'corrupted_count': 2}


In [2]:
def stream_batches(data, batch_size):
    batch = []

    for item in data:
        batch.append(item)

        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch


data = [1, 2, 3, 4, 5, 6, 7, 8]
batch_size = 3

gen = stream_batches(data, batch_size)

print(list(gen))

[[1, 2, 3], [4, 5, 6], [7, 8]]


In [3]:
import re

def extract_contacts(text):
    email_pattern = r'[\w.-]+@[\w.-]+\.\w+'
    phone_pattern = r'(?:\+91\s?)?[6-9]\d{9}'

    emails = re.findall(email_pattern, text)
    phones = re.findall(phone_pattern, text)

    return {
        "emails": emails,
        "phones": phones
    }


text = "Contact support at ravi.kumar@techcorp.in or admin@sales.co. Direct helpline: +91 9876543210 or call 8765432109."

print(extract_contacts(text))

{'emails': ['ravi.kumar@techcorp.in', 'admin@sales.co'], 'phones': ['+91 9876543210', '8765432109']}


In [4]:
class Transaction:

    def __init__(self, txn_id, customer, price, qty, discount):
        self.txn_id = txn_id
        self.customer = customer
        self.price = price
        self.qty = qty
        self.discount = discount

        self.__net_total = (price * qty) * (1 - discount)

    def get_net_total(self):
        return self.__net_total

    def to_dict(self):
        return {
            "txn_id": self.txn_id,
            "customer": self.customer,
            "net_total": self.__net_total
        }


txn = Transaction("TXN_501", "Asha", 500.0, 2, 0.10)

print(txn.get_net_total())
print(txn.to_dict())

900.0
{'txn_id': 'TXN_501', 'customer': 'Asha', 'net_total': 900.0}


In [5]:
import pandas as pd

def analyze_sales_data(raw_records):

    df = pd.DataFrame(raw_records)

    df = df.drop_duplicates()

    df["price"] = df["price"].fillna(df["price"].mean())

    df["revenue"] = df["price"] * df["quantity"]

    result = df.groupby("category")["revenue"].agg(
        total_revenue="sum",
        avg_revenue="mean"
    ).reset_index()

    return result.to_dict("records")


sample_sales = [
    {"txn_id": "T1", "category": "Electronics", "price": 1000.0, "quantity": 2},
    {"txn_id": "T2", "category": "Furniture", "price": 300.0, "quantity": 1},
    {"txn_id": "T3", "category": "Electronics", "price": None, "quantity": 1},
    {"txn_id": "T1", "category": "Electronics", "price": 1000.0, "quantity": 2}
]

print(analyze_sales_data(sample_sales))

ModuleNotFoundError: No module named 'pandas'

In [6]:
import json

def export_kpi_summary(summary_data, top_category, output_filepath):

    report = {
        "status": "SUCCESS",
        "top_category": top_category,
        "metrics": summary_data
    }

    with open(output_filepath, "w") as file:
        json.dump(report, file, indent=4)

    with open(output_filepath, "r") as file:
        return json.load(file)


summary_data = [
    {
        "category": "Electronics",
        "total_revenue": 2650.0,
        "avg_revenue": 1325.0
    },
    {
        "category": "Furniture",
        "total_revenue": 300.0,
        "avg_revenue": 300.0
    }
]

result = export_kpi_summary(
    summary_data,
    "Electronics",
    "kpi_report.json"
)

print(result)

{'status': 'SUCCESS', 'top_category': 'Electronics', 'metrics': [{'category': 'Electronics', 'total_revenue': 2650.0, 'avg_revenue': 1325.0}, {'category': 'Furniture', 'total_revenue': 300.0, 'avg_revenue': 300.0}]}


In [7]:
class InsufficientFundsError(Exception):
    pass


class NegativeAmountError(Exception):
    pass


class BankAccount:

    def __init__(self, account_id, initial_balance):
        self.account_id = account_id
        self.__balance = initial_balance

    def deposit(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Amount must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Amount must be positive")

        if amount > self.__balance:
            raise InsufficientFundsError("Insufficient funds")

        self.__balance -= amount

    def get_balance(self):
        return self.__balance


acc = BankAccount("ACC1001", 500.0)

acc.deposit(200.0)
acc.withdraw(300.0)

print(acc.get_balance())

400.0


In [8]:
from collections import OrderedDict

class DataBuffer:

    def __init__(self, capacity):
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None

        value = self.data.pop(key)
        self.data[key] = value

        return value

    def put(self, key, value):
        if key in self.data:
            self.data.pop(key)

        self.data[key] = value

        if len(self.data) > self.capacity:
            self.data.popitem(last=False)


buf = DataBuffer(2)

buf.put("sensor1", 24.5)
buf.put("sensor2", 28.0)

buf.get("sensor1")

buf.put("sensor3", 31.2)

print(buf.get("sensor2"))
print(buf.get("sensor1"))
print(buf.get("sensor3"))

None
24.5
31.2


In [9]:
import numpy as np

def detect_outliers_zscore(values, threshold=2.0):

    values = np.array(values)

    mean = np.mean(values)
    std = np.std(values)

    z_scores = abs((values - mean) / std)

    return values[z_scores > threshold].tolist()


metrics = [10.0, 12.0, 12.0, 13.0, 12.0,
           11.0, 14.0, 100.0, 12.0]

outliers = detect_outliers_zscore(metrics, threshold=2.0)

print(outliers)

ModuleNotFoundError: No module named 'numpy'

In [10]:
def top_k_frequent_logs(logs, k):

    count = {}

    for log in logs:
        tag = log.split(":")[0]

        if tag in count:
            count[tag] += 1
        else:
            count[tag] = 1

    sorted_tags = sorted(
        count,
        key=lambda x: (-count[x], x)
    )

    return sorted_tags[:k]


logs = [
    "ERROR: db timeout",
    "INFO: user login",
    "ERROR: auth failed",
    "WARNING: disk low",
    "ERROR: lost connection",
    "INFO: user logout"
]

print(top_k_frequent_logs(logs, k=2))

['ERROR', 'INFO']
